In [1]:
import torch
from pathlib import Path

from reasoning_from_scratch.qwen3 import Qwen3Tokenizer, download_qwen3_small

In [2]:
print(f"Torch version is {torch.__version__}")
print(f"Is CUDA available: {torch.cuda.is_available()}")

Torch version is 2.11.0+cu128
Is CUDA available: True


In [3]:
download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

print(f"Tokenizer loaded from {tokenizer_path}")
print(f"Vocab size: {len(tokenizer._tok.get_vocab())}")

Tokenizer loaded from qwen3/tokenizer-base.json
Vocab size: 151665


In [ ]:
prompt = "Explain large language models"

input_tokens = tokenizer.encode(prompt)
print(f"Token: \n{input_tokens}")

for token in input_tokens:
    print(f"{token} -> {tokenizer.decode([token])}")

decoded_prompt = tokenizer.decode(input_tokens)
print(f"Decoded tokens: \n{decoded_prompt}")

Token: 
[840, 20772, 3460, 4128, 4119]
840 -> Ex
20772 -> plain
3460 ->  large
4128 ->  language
4119 ->  models
Decoded tokens: 
Explain large language models


In [ ]:
unknown_words_prompt = "Hi, Hwo are you ?"
input_tokens = tokenizer.encode(unknown_words_prompt)

for token in input_tokens:
    print(f"{token} -> {tokenizer.decode([token])}")

print("-" * 80)

known_words_prompt = "Hi, How are you ?"
input_tokens = tokenizer.encode(known_words_prompt)

for token in input_tokens:
    print(f"{token} -> {tokenizer.decode([token])}")

13048 -> Hi
11 -> ,
472 ->  H
1126 -> wo
525 ->  are
498 ->  you
937 ->  ?
--------------------------------------------------------------------------------
13048 -> Hi
11 -> ,
2585 ->  How
525 ->  are
498 ->  you
937 ->  ?


In [16]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [7]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

In [8]:
QWEN_CONFIG_06_B

{'vocab_size': 151936,
 'context_length': 40960,
 'emb_dim': 1024,
 'n_heads': 16,
 'n_layers': 28,
 'hidden_dim': 3072,
 'head_dim': 128,
 'qk_norm': True,
 'n_kv_groups': 8,
 'rope_base': 1000000.0,
 'dtype': torch.bfloat16}

In [9]:
model = Qwen3Model(QWEN_CONFIG_06_B)
model_weights_path = Path("qwen3") / "qwen3-0.6B-base.pth"
model_weights = torch.load(model_weights_path)
model.load_state_dict(model_weights)

<All keys matched successfully>

In [ ]:
def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")
        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")
    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")
    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

In [13]:
device = get_device()
model.to(device)

Using NVIDIA CUDA GPU


Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [26]:
input_prompt = "Explain large language models."
input_tokens = tokenizer.encode(input_prompt)
input_tokens = torch.tensor(input_tokens).unsqueeze(0)
print(f"Shape: {input_tokens.shape}")
input_tokens

Shape: torch.Size([1, 6])


tensor([[  840, 20772,  3460,  4128,  4119,    13]])

In [28]:
with torch.inference_mode():
    output_tokens = model(input_tokens.to(device))

In [33]:
output_tokens.shape     # Sequence, input_len, dictionary_len
output_tokens = output_tokens.squeeze(0)
output_pred = output_tokens[-1]
print(f"Output pred shape: {output_pred.shape}")

Output pred shape: torch.Size([151936])


In [37]:
best_token = torch.argmax(output_pred, dim=-1, keepdim=True)
best_token

tensor([20286], device='cuda:0')

In [50]:
best_pred = tokenizer.decode(best_token.cpu().numpy())
print(f"Best prediction: {best_pred}")

Best prediction:  Large


In [81]:
@torch.inference_mode()
def generate_predictions(
    model,
    input_prompt, 
    max_seq_len, 
    eos_token
):
    
    model.eval()
    input_tokens = torch.tensor(tokenizer.encode(input_prompt)).unsqueeze(0).to(device)   # num_sequence x seq_len
    for _ in range(max_seq_len):
        output = model(input_tokens)  # num_sequence, seq_len, dictionary_size
        #print(f"Output shape: {output.shape}")
        output = output[:, -1]  # num_sequence, dictionary_size
        #print(f"Output shape: {output.shape}")
        next_token = torch.argmax(output, dim=-1, keepdims=True)    # num_sequence, 1
        #print(f"Next Token shape: {next_token.shape}")
    
        if eos_token is not None:
            #print(f"EOS checks: {torch.all(next_token == eos_token).shape}")
            if torch.all(next_token == eos_token):
                break
        
        yield next_token
        
        input_tokens = torch.cat([input_tokens, next_token], dim=1)
        

In [93]:
for output_token in generate_predictions(model, "Explain me large language models.", 100, None):
    output_token = output_token.squeeze(0).cpu().numpy()
    decoded_output_token = tokenizer.decode(output_token)
    print(decoded_output_token, end="", flush=True)

 Large language models

 are a type of artificial intelligence (AI) that are designed to understand and generate human-like text. They are trained on large amounts of text data, which allows them to learn patterns and relationships between words and phrases. Large language models are often used in a variety of applications, including natural language processing, chatbots, and language translation.

One of the key features of large language models is their ability to generate coherent and contextually relevant text. They can understand the context of a conversation and

In [ ]:
for output_token in generate_predictions(model, "Explain me large language models.", 1000, tokenizer.eos_token_id):
    output_token = output_token.squeeze(0).cpu().numpy()
    decoded_output_token = tokenizer.decode(output_token)
    print(decoded_output_token, end="", flush=True)

 Large language

 models are a type of artificial intelligence (AI) that are designed to understand and generate human-like text. They are trained on large amounts of text data, which allows them to learn patterns and relationships between words and phrases. Large language models are often used in a variety of applications, including natural language processing, chatbots, and language translation.

One of the key features of large language models is their ability to generate coherent and contextually relevant text. They can understand the context of a conversation and generate responses that are consistent with that context. This makes them useful for applications such as virtual assistants, customer service chatbots, and language translation.

Large language models are also capable of understanding and generating complex language structures, such as syntax and semantics. They can understand the meaning of sentences and paragraphs, and generate coherent and contextually relevant text. This makes them u